# 05 — Attach metadata and find useful associations

Attach metadata to each model at the level its outputs naturally represent.

TEMPTED has one score per subject, so it uses metadata that are constant within a subject. MEFISTO has one score per sample, so it uses the full sample-level metadata.

The association table is used only to identify informative plots in notebook 06.

In [1]:
from datetime import datetime
from pathlib import Path

import pandas as pd
from scipy.stats import spearmanr

root = Path(".") if Path("data").exists() else Path("..")
prep = sorted((root / "data" / "preprocessing").iterdir())[-1]
t_dir = sorted((root / "data" / "tempted").iterdir())[-1]
m_dir = sorted((root / "data" / "mefisto").iterdir())[-1]
output = root / "data" / "metadata_analysis" / datetime.now().strftime("%Y%m%d_%H%M%S")
output.mkdir(parents=True)

# load model outputs and full metadata
metadata = pd.read_csv(prep / "metadata.csv", dtype={"sample_id": str, "subject_id": str})
tempted = pd.read_csv(t_dir / "subject_scores.csv", dtype={"subject_id": str})
mefisto = pd.read_csv(m_dir / "sample_factors.csv", dtype={"sample_id": str, "subject_id": str})

t_dims = [c for c in tempted if c.startswith("component_")]
m_dims = [c for c in mefisto if c.startswith("factor_")]

In [2]:
# find metadata that are constant within each subject
constant = [
    c for c in metadata.columns
    if c not in ["sample_id", "subject_id"]
    and metadata.groupby("subject_id")[c].nunique().max() <= 1
]

# attach subject-level metadata to tempted
subject_metadata = metadata[["subject_id"] + constant].drop_duplicates("subject_id")
tempted = tempted.merge(subject_metadata, on="subject_id", how="left")

# attach full sample-level metadata to mefisto
mefisto = mefisto.merge(
    metadata,
    on=["sample_id", "subject_id", "age"],
    how="left",
)

# save metadata-rich model tables
tempted.to_csv(output / "tempted_with_metadata.csv", index=False)
mefisto.to_csv(output / "mefisto_with_metadata.csv", index=False)

In [3]:
# score useful metadata relationships for plotting
rows = []
ignore = {"sample_id", "subject_id", "SampleID", "subjectID"}

for method, frame, dims in [
    ("TEMPTED", tempted, t_dims),
    ("MEFISTO", mefisto, m_dims),
]:
    metadata_columns = [
        c for c in frame.columns
        if c not in ignore and c not in dims
    ]

    for variable in metadata_columns:
        values = frame[variable]
        numeric = pd.to_numeric(values, errors="coerce")

        # numeric metadata use spearman correlation
        if numeric.notna().sum() >= 10 and numeric.nunique() >= 5:
            for dim in dims:
                keep = numeric.notna() & frame[dim].notna()
                rho = spearmanr(
                    numeric[keep],
                    frame.loc[keep, dim],
                ).statistic

                rows.append([
                    method,
                    variable,
                    "numeric",
                    dim,
                    rho,
                    abs(rho),
                ])

            continue

        # categorical metadata use the spread in standardized group means
        counts = values.value_counts(dropna=True)
        groups = counts[counts >= 5].index

        if 2 <= len(groups) <= 8:
            temp = frame[values.isin(groups)]

            for dim in dims:
                z = (temp[dim] - temp[dim].mean()) / temp[dim].std(ddof=0)
                means = z.groupby(temp[variable]).mean()
                spread = means.max() - means.min()

                rows.append([
                    method,
                    variable,
                    "categorical",
                    dim,
                    spread,
                    abs(spread),
                ])

associations = pd.DataFrame(
    rows,
    columns=[
        "method",
        "metadata",
        "type",
        "dimension",
        "association",
        "absolute_association",
    ],
)

associations.to_csv(output / "metadata_associations.csv", index=False)

print("Saved:", output)
associations.sort_values("absolute_association", ascending=False).head(20)

Saved: ../data/metadata_analysis/20260812_013935


,method,metadata,type,dimension,association,absolute_association
7,TEMPTED,country,categorical,component_2,1.257660,1.257660
70,MEFISTO,Any_baby_formula,categorical,factor_1,1.161125,1.161125
68,MEFISTO,partly_hydrosylated_formula,categorical,factor_1,1.137809,1.137809
64,MEFISTO,Regular_formula,categorical,factor_1,1.063412,1.063412
66,MEFISTO,hydrosylated_formula,categorical,factor_1,0.983077,0.983077
98,MEFISTO,Milk,categorical,factor_1,0.913709,0.913709
84,MEFISTO,Rye,categorical,factor_1,0.902722,0.902722
94,MEFISTO,Eggs,categorical,factor_1,0.877263,0.877263
92,MEFISTO,Vegetables,categorical,factor_1,0.842237,0.842237
80,MEFISTO,Oat,categorical,factor_1,0.835810,0.835810
